<a href="https://colab.research.google.com/github/MoeTahech/real-fake-face-detection-thesis/blob/main/notebooks/04_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Phase 4: Ensemble (ResNet18 + EfficientNet-B0)

Trains a second, independent architecture (EfficientNet-B0) the same way we
fine-tuned ResNet18 in Phase 2 (unfreeze only the last block), then combines
both models' predictions (simple probability averaging) into an ensemble.

Crucially, the ensemble is evaluated on **both**:
- the in-distribution test set (same as Phases 1-2), and
- the cross-dataset (face-swap) test set from Phase 3

...so we can honestly see whether ensembling actually improves generalization,
or only pads the in-distribution number.

Requires `01_baseline.ipynb` and `02_finetuning.ipynb` to have been run at least
once (so `best_model_finetuned.pth` exists in Drive).

Runtime → Change runtime type → GPU, then run cells top to bottom.

### Step 1 — Confirm you have a GPU

In [ ]:
!nvidia-smi

Mon Jul 27 03:27:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/thesis_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to / loaded from:", CHECKPOINT_DIR)

Mounted at /content/drive
Checkpoints will be saved to / loaded from: /content/drive/MyDrive/thesis_checkpoints


### Step 3 — Kaggle API access

In [ ]:
import os

os.environ["KAGGLE_API_TOKEN"] = "YOUR_KAGGLE_API_TOKEN_HERE"

### Step 4 — Download and link the primary dataset (140k Real and Fake Faces)

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q 140k-real-and-fake-faces.zip -d data_raw
!find data_raw -maxdepth 4 -type d

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:42<00:00, 95.6MB/s]

data_raw
data_raw/real_vs_fake
data_raw/real_vs_fake/real-vs-fake
data_raw/real_vs_fake/real-vs-fake/train
data_raw/real_vs_fake/real-vs-fake/train/real
data_raw/real_vs_fake/real-vs-fake/train/fake
data_raw/real_vs_fake/real-vs-fake/valid
data_raw/real_vs_fake/real-vs-fake/valid/real
data_raw/real_vs_fake/real-vs-fake/valid/fake
data_raw/real_vs_fake/real-vs-fake/test
data_raw/real_vs_fake/real-vs-fake/test/real
data_raw/real_vs_fake/real-vs-fake/test/fake


In [ ]:
import os

def find_split_root(root="data_raw"):
    for dirpath, dirnames, filenames in os.walk(root):
        if "train" in dirnames:
            train_path = os.path.join(dirpath, "train")
            if os.path.isdir(os.path.join(train_path, "real")) and os.path.isdir(os.path.join(train_path, "fake")):
                return dirpath
    return None

base = find_split_root("data_raw")
base_abs = os.path.abspath(base)
print("Found dataset root:", base_abs)

os.makedirs("data", exist_ok=True)
val_name = "valid" if os.path.isdir(os.path.join(base, "valid")) else "val"

!ln -sfn "{base_abs}/train" data/train
!ln -sfn "{base_abs}/{val_name}" data/val

print("train/real:", len(os.listdir("data/train/real")))
print("train/fake:", len(os.listdir("data/train/fake")))

Found dataset root: /content/data_raw/real_vs_fake/real-vs-fake
train/real: 50000
train/fake: 50000


### Step 5 (optional, recommended) — Small subset for a fast sanity check

In [ ]:
import os, random, shutil

def make_subset(src_split, dst_split, n_per_class):
    for cls in ["real", "fake"]:
        src_dir = f"data/{src_split}/{cls}"
        dst_dir = f"data_small/{dst_split}/{cls}"
        os.makedirs(dst_dir, exist_ok=True)
        files = os.listdir(src_dir)
        random.shuffle(files)
        for f in files[:n_per_class]:
            shutil.copy(os.path.join(src_dir, f), os.path.join(dst_dir, f))

make_subset("train", "train", 1000)
make_subset("val", "val", 250)
print("Small subset ready in ./data_small")

Small subset ready in ./data_small


### Step 6 — Write the EfficientNet-B0 training script

Same philosophy as Phase 2's ResNet18 fine-tuning: freeze most of the network,
unfreeze only the last feature block + the new classification head. This keeps
the comparison with ResNet18 fair (both are "last-block fine-tuned" models).

In [ ]:
%%writefile train_efficientnet.py
"""
Fine-tune EfficientNet-B0: unfreeze only the last feature block + classifier head.

Usage:
    python train_efficientnet.py --data_dir ./data --epochs 5 --batch_size 64 --lr 1e-4 --checkpoint_dir .
"""
import argparse
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm


def get_dataloaders(data_dir, batch_size, img_size=224):
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ])
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        normalize,
    ])
    train_set = datasets.ImageFolder(f"{data_dir}/train", transform=train_transform)
    val_set = datasets.ImageFolder(f"{data_dir}/val", transform=val_transform)
    print("Class mapping:", train_set.class_to_idx)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, val_loader, train_set.class_to_idx


def build_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze only the last feature block (highest-level features)
    for param in model.features[-1].parameters():
        param.requires_grad = True

    # Replace classifier head (Dropout + Linear) — new Linear is trainable by default
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(in_features, 2)
    )
    return model


def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, desc="train" if train else "val"):
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=str, default="./data")
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--checkpoint_dir", type=str, default=".")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    train_loader, val_loader, class_to_idx = get_dataloaders(args.data_dir, args.batch_size)
    model = build_model().to(device)
    criterion = nn.CrossEntropyLoss()

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable_params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {n_trainable:,} / {n_total:,}")

    optimizer = torch.optim.Adam(trainable_params, lr=args.lr)

    os.makedirs(args.checkpoint_dir, exist_ok=True)
    save_path = os.path.join(args.checkpoint_dir, "best_model_efficientnet.pth")

    best_val_acc = 0.0
    for epoch in range(args.epochs):
        print(f"\nEpoch {epoch + 1}/{args.epochs}")
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
        print(f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({"model_state": model.state_dict(), "class_to_idx": class_to_idx}, save_path)
            print(f"Saved new best model to {save_path} (val_acc={val_acc:.4f})")

    print(f"\nDone. Best validation accuracy: {best_val_acc:.4f}")
    print(f"Final checkpoint is at: {save_path}")


if __name__ == "__main__":
    main()


Writing train_efficientnet.py


### Step 7 — Train EfficientNet-B0 (small subset sanity check, then full run)

In [ ]:
!python train_efficientnet.py --data_dir ./data_small --epochs 3 --batch_size 32 --lr 1e-4 --checkpoint_dir "{CHECKPOINT_DIR}"

Using device: cuda
Class mapping: {'fake': 0, 'real': 1}
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 134MB/s] 
Trainable parameters: 414,722 / 4,010,110

Epoch 1/3
train: 100% 63/63 [00:08<00:00,  7.02it/s]
val: 100% 16/16 [00:01<00:00,  9.51it/s]
train_loss=0.6553 train_acc=0.6220 val_loss=0.5920 val_acc=0.6980
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/best_model_efficientnet.pth (val_acc=0.6980)

Epoch 2/3
train: 100% 63/63 [00:07<00:00,  8.60it/s]
val: 100% 16/16 [00:01<00:00,  8.47it/s]
train_loss=0.5803 train_acc=0.7155 val_loss=0.5405 val_acc=0.7300
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/best_model_efficientnet.pth (val_acc=0.7300)

Epoch 3/3
train: 100% 63/63 [00:05<00:00, 10.66it/s]
val: 100% 16/16 [00:01<00:00, 10.27it/s]
train_loss=0.5331 train_acc=0.7435 val_loss=0.5056 val

In [ ]:
!python train_efficientnet.py --data_dir ./data --epochs 5 --batch_size 64 --lr 1e-4 --checkpoint_dir "{CHECKPOINT_DIR}"

Using device: cuda
Class mapping: {'fake': 0, 'real': 1}
Trainable parameters: 414,722 / 4,010,110

Epoch 1/5
train: 100% 1563/1563 [06:18<00:00,  4.13it/s]
val: 100% 313/313 [01:08<00:00,  4.59it/s]
train_loss=0.4061 train_acc=0.8170 val_loss=0.2961 val_acc=0.8776
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/best_model_efficientnet.pth (val_acc=0.8776)

Epoch 2/5
train: 100% 1563/1563 [05:44<00:00,  4.53it/s]
val: 100% 313/313 [01:05<00:00,  4.82it/s]
train_loss=0.2826 train_acc=0.8833 val_loss=0.2326 val_acc=0.9048
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/best_model_efficientnet.pth (val_acc=0.9048)

Epoch 3/5
train: 100% 1563/1563 [05:43<00:00,  4.55it/s]
val: 100% 313/313 [01:07<00:00,  4.66it/s]
train_loss=0.2323 train_acc=0.9070 val_loss=0.1955 val_acc=0.9217
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/best_model_efficientnet.pth (val_acc=0.9217)

Epoch 4/5
train: 100% 1563/1563 [05:22<00:00,  4.84it/s]
val: 100%

### Step 8 — Download the cross-dataset (face-swap) test set again

Same second dataset used in Phase 3 — needed here so we can test the ensemble's
generalization, not just its in-distribution accuracy.

In [ ]:
!kaggle datasets download -d manjilkarki/deepfake-and-real-images
!unzip -q deepfake-and-real-images.zip -d data_raw_2

def find_real_fake_folders(root):
    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        lower = [d.lower() for d in dirnames]
        if "real" in lower and "fake" in lower:
            candidates.append(dirpath)
    return candidates

candidates = find_real_fake_folders("data_raw_2")
test_candidates = [x for x in candidates if "test" in x.lower()]
CROSS_TEST_DIR = test_candidates[0] if test_candidates else candidates[0]
print("Cross-dataset test folder:", CROSS_TEST_DIR)

Dataset URL: https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images
License(s): unknown
100% 1.68G/1.68G [00:17<00:00, 104MB/s]

Cross-dataset test folder: data_raw_2/Dataset/Test


### Step 9 — Build the ensemble and evaluate on BOTH test sets

Loads ResNet18 (fine-tuned, Phase 2) and EfficientNet-B0 (this phase), averages
their predicted probabilities, and evaluates individually + as an ensemble on:
1. the in-distribution test set (same source as training)
2. the cross-dataset test set (unseen face-swap source, from Phase 3)

This directly answers whether ensembling helps generalization or only helps
in-distribution.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), normalize])


def load_resnet(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()
    return model, checkpoint["class_to_idx"]


def load_efficientnet(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(p=0.2), nn.Linear(in_features, 2))
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()
    return model, checkpoint["class_to_idx"]


resnet_model, resnet_class_to_idx = load_resnet(f"{CHECKPOINT_DIR}/best_model_finetuned.pth")
effnet_model, effnet_class_to_idx = load_efficientnet(f"{CHECKPOINT_DIR}/best_model_efficientnet.pth")
# Both were trained on the same dataset, so class_to_idx should match — sanity check:
assert resnet_class_to_idx == effnet_class_to_idx, "Class mappings differ between models!"
real_idx = resnet_class_to_idx["real"]
print("Shared class mapping:", resnet_class_to_idx)


def evaluate_all(test_dir, dataset_label, remap_by_name=False):
    test_set = datasets.ImageFolder(test_dir, transform=transform)
    loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=2)
    print(f"\n{dataset_label} — size: {len(test_set)}, on-disk mapping: {test_set.class_to_idx}")

    if remap_by_name:
        remap = {idx: resnet_class_to_idx[name.lower()] for name, idx in test_set.class_to_idx.items()}
    else:
        remap = {idx: idx for idx in test_set.class_to_idx.values()}

    all_labels, resnet_probs, effnet_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            r_probs = torch.softmax(resnet_model(images), dim=1)[:, real_idx].cpu().numpy()
            e_probs = torch.softmax(effnet_model(images), dim=1)[:, real_idx].cpu().numpy()
            resnet_probs.extend(r_probs)
            effnet_probs.extend(e_probs)
            all_labels.extend([remap[l.item()] for l in labels])

    import numpy as np
    resnet_probs = np.array(resnet_probs)
    effnet_probs = np.array(effnet_probs)
    ensemble_probs = (resnet_probs + effnet_probs) / 2.0

    results = {}
    for name, probs in [("ResNet18 (Phase 2)", resnet_probs),
                         ("EfficientNet-B0", effnet_probs),
                         ("Ensemble (avg)", ensemble_probs)]:
        preds = (probs >= 0.5).astype(int)
        acc = accuracy_score(all_labels, preds)
        auc = roc_auc_score(all_labels, probs)
        results[name] = (acc, auc)
        print(f"  {name:<22} Accuracy={acc:.4f}  AUC={auc:.4f}")

    return results


print("=" * 60)
print("IN-DISTRIBUTION TEST SET (same source as training)")
print("=" * 60)
in_dist_results = evaluate_all("data_raw/real_vs_fake/real-vs-fake/test", "In-distribution")

print("\n" + "=" * 60)
print("CROSS-DATASET TEST SET (unseen face-swap source)")
print("=" * 60)
cross_results = evaluate_all(CROSS_TEST_DIR, "Cross-dataset", remap_by_name=True)

print("\n\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"{'Model':<22} {'In-dist Acc':<13} {'In-dist AUC':<13} {'Cross Acc':<12} {'Cross AUC':<12}")
for name in ["ResNet18 (Phase 2)", "EfficientNet-B0", "Ensemble (avg)"]:
    ia, iu = in_dist_results[name]
    ca, cu = cross_results[name]
    print(f"{name:<22} {ia:<13.4f} {iu:<13.4f} {ca:<12.4f} {cu:<12.4f}")

Shared class mapping: {'fake': 0, 'real': 1}
IN-DISTRIBUTION TEST SET (same source as training)

In-distribution — size: 20000, on-disk mapping: {'fake': 0, 'real': 1}
  ResNet18 (Phase 2)     Accuracy=0.9881  AUC=0.9993
  EfficientNet-B0        Accuracy=0.9382  AUC=0.9871
  Ensemble (avg)         Accuracy=0.9899  AUC=0.9992

CROSS-DATASET TEST SET (unseen face-swap source)

Cross-dataset — size: 10905, on-disk mapping: {'Fake': 0, 'Real': 1}
  ResNet18 (Phase 2)     Accuracy=0.4945  AUC=0.5253
  EfficientNet-B0        Accuracy=0.4888  AUC=0.4993
  Ensemble (avg)         Accuracy=0.4935  AUC=0.4980


SUMMARY
Model                  In-dist Acc   In-dist AUC   Cross Acc    Cross AUC   
ResNet18 (Phase 2)     0.9881        0.9993        0.4945       0.5253      
EfficientNet-B0        0.9382        0.9871        0.4888       0.4993      
Ensemble (avg)         0.9899        0.9992        0.4935       0.4980      


---
### What to save for this phase
- EfficientNet-B0's own training curve and trainable parameter count (Step 7)
- The full summary table from Step 9 — five numbers per model (in-dist acc/AUC,
  cross-dataset acc/AUC), for all three: ResNet18 alone, EfficientNet alone, and
  the ensemble

**What to look for:** if the ensemble's cross-dataset AUC is close to the
individual models' (~0.5), that confirms ensembling two models trained on the
same single-source data does not fix the generalization gap — both models
likely share the same blind spot. If it's noticeably higher, that's a genuinely
interesting positive result worth highlighting. Either outcome is a valid,
reportable finding — send me the numbers either way.